In [20]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


### set up pyspark session

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/31 15:38:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### set up config

In [3]:
# set up config
model_train_date_str = "2024-06-01"
train_test_period_months = 9
oot_period_months = 2
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] =  oot_period_months
config["model_train_date"] =  datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] =  config['model_train_date'] - timedelta(days = 1)
config["oot_start_date"] =  config['model_train_date'] - relativedelta(months = oot_period_months)
config["train_test_end_date"] =  config["oot_start_date"] - timedelta(days = 1)
config["train_test_start_date"] =  config["oot_start_date"] - relativedelta(months = train_test_period_months)
config["train_test_ratio"] = train_test_ratio 


pprint.pprint(config)

{'model_train_date': datetime.datetime(2024, 6, 1, 0, 0),
 'model_train_date_str': '2024-06-01',
 'oot_end_date': datetime.datetime(2024, 5, 31, 0, 0),
 'oot_period_months': 2,
 'oot_start_date': datetime.datetime(2024, 4, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 3, 31, 0, 0),
 'train_test_period_months': 9,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)}


### get label store

In [4]:
# connect to label store
folder_path = "datamart/gold/label_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
label_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",label_store_sdf.count())

label_store_sdf.show()

row_count: 12500
+--------------------+-----------+-----+----------+---------------+-------------+
|             loan_id|Customer_ID|label| label_def|loan_start_date|snapshot_date|
+--------------------+-----------+-----+----------+---------------+-------------+
|CUS_0x10ac_2024_0...| CUS_0x10ac|    0|30dpd_6mob|     2024-08-01|   2025-02-01|
|CUS_0x10c5_2024_0...| CUS_0x10c5|    1|30dpd_6mob|     2024-08-01|   2025-02-01|
|CUS_0x1145_2024_0...| CUS_0x1145|    1|30dpd_6mob|     2024-08-01|   2025-02-01|
|CUS_0x11ac_2024_0...| CUS_0x11ac|    0|30dpd_6mob|     2024-08-01|   2025-02-01|
|CUS_0x122c_2024_0...| CUS_0x122c|    0|30dpd_6mob|     2024-08-01|   2025-02-01|
|CUS_0x1274_2024_0...| CUS_0x1274|    1|30dpd_6mob|     2024-08-01|   2025-02-01|
|CUS_0x1288_2024_0...| CUS_0x1288|    1|30dpd_6mob|     2024-08-01|   2025-02-01|
|CUS_0x12cc_2024_0...| CUS_0x12cc|    1|30dpd_6mob|     2024-08-01|   2025-02-01|
|CUS_0x1338_2024_0...| CUS_0x1338|    0|30dpd_6mob|     2024-08-01|   2025-02-01|

In [5]:
# extract label store
labels_sdf = label_store_sdf.filter((col("snapshot_date") >= config["train_test_start_date"]) & (col("snapshot_date") <= config["oot_end_date"]))

print("extracted labels_sdf", labels_sdf.count(), "between", config["train_test_start_date"], "and", config["oot_end_date"])

[Stage 6:================>                                          (2 + 5) / 7]

extracted labels_sdf 5469 between 2023-07-01 00:00:00 and 2024-05-31 00:00:00


In [6]:
labels_sdf.show()

+--------------------+-----------+-----+----------+---------------+-------------+
|             loan_id|Customer_ID|label| label_def|loan_start_date|snapshot_date|
+--------------------+-----------+-----+----------+---------------+-------------+
|CUS_0x1037_2023_0...| CUS_0x1037|    0|30dpd_6mob|     2023-01-01|   2023-07-01|
|CUS_0x1069_2023_0...| CUS_0x1069|    0|30dpd_6mob|     2023-01-01|   2023-07-01|
|CUS_0x114a_2023_0...| CUS_0x114a|    0|30dpd_6mob|     2023-01-01|   2023-07-01|
|CUS_0x1184_2023_0...| CUS_0x1184|    0|30dpd_6mob|     2023-01-01|   2023-07-01|
|CUS_0x1297_2023_0...| CUS_0x1297|    1|30dpd_6mob|     2023-01-01|   2023-07-01|
|CUS_0x12fb_2023_0...| CUS_0x12fb|    0|30dpd_6mob|     2023-01-01|   2023-07-01|
|CUS_0x1325_2023_0...| CUS_0x1325|    0|30dpd_6mob|     2023-01-01|   2023-07-01|
|CUS_0x1341_2023_0...| CUS_0x1341|    0|30dpd_6mob|     2023-01-01|   2023-07-01|
|CUS_0x1375_2023_0...| CUS_0x1375|    1|30dpd_6mob|     2023-01-01|   2023-07-01|
|CUS_0x13a8_2023

### get features

In [7]:
# connect to feature store - clickstream
folder_path = "datamart/gold/feature_store_clickstream/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
feature_clickstream_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",feature_clickstream_store_sdf.count())

feature_clickstream_store_sdf.show()

row_count: 215376
+-----------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-------------+
|Customer_ID|avg_fe_1|avg_fe_2|avg_fe_3|avg_fe_4|avg_fe_5|avg_fe_6|avg_fe_7|avg_fe_8|avg_fe_9|avg_fe_10|avg_fe_11|avg_fe_12|avg_fe_13|avg_fe_14|avg_fe_15|avg_fe_16|avg_fe_17|avg_fe_18|avg_fe_19|avg_fe_20|snapshot_date|
+-----------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-------------+
| CUS_0x6ba9|    84.0|     0.0|   213.5|   185.0|   117.0|    51.5|    90.0|    77.5|    41.0|    191.5|      0.0|    106.5|    117.0|     52.0|     22.5|     23.0|    197.5|    161.5|     90.5|    293.0|   2023-02-01|
| CUS_0x74c7|   233.0|    54.5|   245.0|    93.5|   118.5|    94.0|    19.0|   144.5|   129.0|    202.5|  

In [8]:
# connect to feature store - financials
folder_path = "datamart/gold/feature_store_financials/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
feature_financials_store_sdf = spark.read.option("header", "true").parquet(*files_list)

print("row_count:",feature_financials_store_sdf.count())

feature_financials_store_sdf.show()

row_count: 12500
+-----------+-------------+------------+--------------+---------------------+----------------+--------------------+------------------------+
|Customer_ID|snapshot_date|Num_Fin_Pdts|Debt_to_Salary|Loans_per_Credit_Item|Outstanding_Debt|Changed_Credit_Limit|Credit_History_Age_Month|
+-----------+-------------+------------+--------------+---------------------+----------------+--------------------+------------------------+
| CUS_0x10ac|   2024-08-01|          14|     0.6713313|           0.36363637|          853.41|               18.45|                     195|
| CUS_0x10c5|   2024-08-01|          10|   0.086091995|                  0.1|         1134.83|                6.04|                     362|
| CUS_0x1145|   2024-08-01|          20|     0.8735474|                 0.75|          1263.1|                 8.9|                     149|
| CUS_0x11ac|   2024-08-01|           7|    0.08111457|                  0.0|          478.85|                7.18|                     2

In [9]:
# extract label store
feature_clickstream_sdf = feature_clickstream_store_sdf.filter((col("snapshot_date") <= config["oot_end_date"]))

print("extracted feature_clickstream_sdf", feature_clickstream_sdf.count(), "before", config["oot_end_date"])

[Stage 20:===========================================>              (6 + 2) / 8]

extracted feature_clickstream_sdf 152558 before 2024-05-31 00:00:00


In [10]:
# extract label store
feature_financials_sdf = feature_financials_store_sdf.filter((col("snapshot_date") <= config["oot_end_date"]))

print("extracted feature_clickstream_sdf", feature_financials_sdf.count(), "before", config["oot_end_date"])

[Stage 23:========>                                                 (1 + 6) / 7]

extracted feature_clickstream_sdf 8476 before 2024-05-31 00:00:00


### prepare data for modeling

In [11]:
# prepare data for modeling
data_pdf = labels_sdf.alias("a").join(
        feature_clickstream_sdf.alias("b"), 
        (F.col("a.Customer_ID") == F.col("b.Customer_ID")) & (F.col("a.loan_start_date") == F.col("b.snapshot_date")), how="left").drop(
    F.col("b.snapshot_date"), F.col("b.Customer_ID"))

data_pdf = data_pdf.alias("a").join(
        feature_financials_sdf.alias("c"), 
        (F.col("a.Customer_ID") == F.col("c.Customer_ID")) & (F.col("a.loan_start_date") == F.col("c.snapshot_date")), how="left").drop(
    F.col("c.snapshot_date"), F.col("c.Customer_ID"))

data_pdf = data_pdf.toPandas()
data_pdf["Outstanding_Debt_log"] = np.log1p(data_pdf["Outstanding_Debt"])
data_pdf = data_pdf.drop('Outstanding_Debt', axis=1)
data_pdf

,loan_id,Customer_ID,label,label_def,loan_start_date,snapshot_date,avg_fe_1,avg_fe_2,avg_fe_3,avg_fe_4,...,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20,Num_Fin_Pdts,Debt_to_Salary,Loans_per_Credit_Item,Changed_Credit_Limit,Credit_History_Age_Month,Outstanding_Debt_log
0,CUS_0x10ff_2023_10_01,CUS_0x10ff,0,30dpd_6mob,2023-10-01,2024-04-01,114.166667,157.166667,64.666667,135.500000,...,85.833333,182.833333,162.500000,106.666667,10.0,0.528732,0.222222,NaN,205,6.529492
1,CUS_0x1130_2023_07_01,CUS_0x1130,0,30dpd_6mob,2023-07-01,2024-01-01,141.333333,40.000000,61.833333,114.000000,...,130.666667,133.666667,129.333333,102.833333,13.0,0.099550,0.000000,9.250000,329,6.963474
2,CUS_0x1136_2023_06_01,CUS_0x1136,1,30dpd_6mob,2023-06-01,2023-12-01,96.166667,188.833333,30.666667,219.666667,...,137.333333,145.500000,126.000000,77.333333,17.0,0.443647,0.384615,NaN,142,7.148527
3,CUS_0x113e_2023_02_01,CUS_0x113e,1,30dpd_6mob,2023-02-01,2023-08-01,208.500000,245.500000,154.000000,162.000000,...,104.000000,133.000000,39.500000,249.500000,21.0,0.727757,0.571429,4.480000,138,7.589654
4,CUS_0x117d_2023_09_01,CUS_0x117d,0,30dpd_6mob,2023-09-01,2024-03-01,108.500000,64.166667,118.166667,103.166667,...,163.000000,100.833333,91.833333,95.166667,9.0,0.228633,0.666667,3.610000,257,7.201402
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5464,CUS_0xf4d_2023_03_01,CUS_0xf4d,0,30dpd_6mob,2023-03-01,2023-09-01,180.666667,35.333333,162.333333,92.333333,...,97.000000,108.333333,40.666667,108.333333,14.0,0.453449,0.363636,12.370000,147,6.573038
5465,CUS_0xfa7_2023_03_01,CUS_0xfa7,0,30dpd_6mob,2023-03-01,2023-09-01,135.666667,101.333333,40.666667,46.000000,...,149.666667,85.666667,81.666667,184.333333,13.0,0.055332,0.400000,1.380000,287,6.332249
5466,CUS_0xfae_2023_03_01,CUS_0xfae,1,30dpd_6mob,2023-03-01,2023-09-01,35.333333,144.666667,37.333333,88.333333,...,171.666667,53.666667,179.666667,213.000000,18.0,1.268101,0.357143,2.500000,72,8.238629
5467,CUS_0xfb8_2023_07_01,CUS_0xfb8,0,30dpd_6mob,2023-07-01,2024-01-01,132.000000,89.000000,111.333333,118.833333,...,93.000000,137.833333,101.666667,85.166667,8.0,0.222858,0.800000,18.799999,288,6.470598


In [12]:
# split data into train - test - oot
oot_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["oot_start_date"].date()) & (data_pdf['snapshot_date'] <= config["oot_end_date"].date())]
train_test_pdf = data_pdf[(data_pdf['snapshot_date'] >= config["train_test_start_date"].date()) & (data_pdf['snapshot_date'] <= config["train_test_end_date"].date())]

feature_cols = [fe_col for fe_col in data_pdf.columns if fe_col.startswith('avg_fe_') or 
                fe_col in ['Num_Fin_Pdts','Debt_to_Salary','Loans_per_Credit_Item','Outstanding_Debt_log','Changed_Credit_Limit','Credit_History_Age_Month']]

feature_cols

['avg_fe_1',
 'avg_fe_2',
 'avg_fe_3',
 'avg_fe_4',
 'avg_fe_5',
 'avg_fe_6',
 'avg_fe_7',
 'avg_fe_8',
 'avg_fe_9',
 'avg_fe_10',
 'avg_fe_11',
 'avg_fe_12',
 'avg_fe_13',
 'avg_fe_14',
 'avg_fe_15',
 'avg_fe_16',
 'avg_fe_17',
 'avg_fe_18',
 'avg_fe_19',
 'avg_fe_20',
 'Num_Fin_Pdts',
 'Debt_to_Salary',
 'Loans_per_Credit_Item',
 'Changed_Credit_Limit',
 'Credit_History_Age_Month',
 'Outstanding_Debt_log']

In [13]:
X_oot = oot_pdf[feature_cols]
y_oot = oot_pdf["label"]
X_train, X_test, y_train, y_test = train_test_split(
    train_test_pdf[feature_cols], train_test_pdf["label"], 
    test_size= 1 - config["train_test_ratio"],
    random_state=88,     # Ensures reproducibility
    shuffle=True,        # Shuffle the data before splitting
    stratify=train_test_pdf["label"]           # Stratify based on the label column
)


print('X_train', X_train.shape[0])
print('X_test', X_test.shape[0])
print('X_oot', X_oot.shape[0])
print('y_train', y_train.shape[0], round(y_train.mean(),2))
print('y_test', y_test.shape[0], round(y_test.mean(),2))
print('y_oot', y_oot.shape[0], round(y_oot.mean(),2))

X_train

X_train 3592
X_test 899
X_oot 978
y_train 3592 0.28
y_test 899 0.28
y_oot 978 0.27


,avg_fe_1,avg_fe_2,avg_fe_3,avg_fe_4,avg_fe_5,avg_fe_6,avg_fe_7,avg_fe_8,avg_fe_9,avg_fe_10,...,avg_fe_17,avg_fe_18,avg_fe_19,avg_fe_20,Num_Fin_Pdts,Debt_to_Salary,Loans_per_Credit_Item,Changed_Credit_Limit,Credit_History_Age_Month,Outstanding_Debt_log
5094,150.166667,128.666667,44.166667,55.333333,74.000000,144.833333,76.333333,53.000000,143.000000,75.833333,...,79.833333,58.666667,93.166667,118.666667,16.0,0.835437,0.133333,8.280000,110,7.827611
5087,52.000000,166.000000,128.000000,52.000000,251.000000,99.000000,118.333333,94.000000,122.333333,28.666667,...,74.333333,135.333333,180.666667,69.666667,NaN,0.537265,NaN,2.240000,128,7.844962
164,49.000000,62.166667,67.166667,128.833333,109.333333,115.833333,141.000000,121.833333,154.500000,145.666667,...,96.833333,126.500000,91.166667,40.333333,15.0,0.050898,0.333333,2.020000,204,5.832058
817,117.500000,183.250000,17.500000,141.750000,164.000000,94.500000,63.750000,144.750000,112.500000,108.000000,...,95.750000,205.500000,76.750000,117.500000,18.0,1.786735,0.583333,17.670000,74,7.538218
1431,100.333333,199.333333,96.666667,66.166667,169.666667,139.000000,164.333333,120.000000,119.333333,86.000000,...,99.000000,112.166667,182.833333,168.000000,14.0,0.189314,0.363636,6.210000,291,7.210242
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2401,29.000000,182.666667,226.666667,166.000000,40.666667,50.000000,116.000000,104.000000,57.333333,3.000000,...,13.333333,69.666667,175.333333,111.666667,18.0,1.330196,0.266667,8.940000,227,7.715016
1604,120.666667,54.000000,78.666667,167.666667,163.333333,112.333333,126.333333,172.666667,59.333333,136.000000,...,4.333333,87.333333,115.000000,99.000000,16.0,1.689042,0.133333,18.139999,224,7.675787
2022,101.333333,100.500000,126.666667,83.166667,203.500000,130.666667,90.666667,120.166667,161.000000,78.500000,...,122.666667,82.833333,115.500000,121.833333,19.0,0.515141,0.333333,11.740000,240,7.710124
2455,92.833333,135.833333,105.666667,66.666667,144.166667,84.333333,143.666667,76.166667,145.500000,151.333333,...,112.666667,39.000000,111.500000,117.000000,13.0,0.098977,0.076923,7.780000,382,6.157614


### preprocess data

In [14]:
# set up standard scalar preprocessing
scaler = StandardScaler()

transformer_stdscaler = scaler.fit(X_train) # Q which should we use? train? test? oot? all?

# transform data
X_train_processed = transformer_stdscaler.transform(X_train)
X_test_processed = transformer_stdscaler.transform(X_test)
X_oot_processed = transformer_stdscaler.transform(X_oot)

print('X_train_processed', X_train_processed.shape[0])
print('X_test_processed', X_test_processed.shape[0])
print('X_oot_processed', X_oot_processed.shape[0])

pd.DataFrame(X_train_processed)

X_train_processed 3592
X_test_processed 899
X_oot_processed 978


,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,0.775137,0.337021,-1.301609,-1.118946,-0.800125,0.685510,-0.765406,-1.267520,0.419887,-0.928224,...,-0.568803,-1.014503,-0.310788,0.171154,0.472956,0.098895,-0.843042,-0.355620,-1.132744,0.914533
1,-1.175827,1.078831,0.294448,-1.183674,2.757594,-0.223289,0.056427,-0.476857,0.033012,-1.826967,...,-0.677587,0.512934,1.408562,-0.768174,NaN,-0.166795,NaN,-1.258673,-0.952933,0.930953
2,-1.235449,-0.984329,-0.863725,0.308309,-0.089921,0.110488,0.499956,0.059895,0.635164,0.402422,...,-0.232561,0.336947,-0.350088,-1.330494,0.264145,-0.600180,0.065184,-1.291566,-0.193730,-0.974001
3,0.125920,1.421587,-1.809301,0.559130,1.008885,-0.312516,-1.011630,0.501831,-0.151066,-0.315301,...,-0.253989,1.910872,-0.633371,0.148789,0.890578,0.946562,1.200466,1.048299,-1.492366,0.640659
4,-0.215250,1.741162,-0.302090,-0.908580,1.122785,0.569845,0.956530,0.024540,-0.023148,-0.734503,...,-0.189707,0.051382,1.451136,1.116873,0.055334,-0.476842,0.202794,-0.665110,0.675357,0.330272
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3587,-1.632928,1.409997,2.172909,1.030027,-1.470129,-1.194877,0.010770,-0.284012,-1.183773,-2.316035,...,-1.884100,-0.795349,1.303764,0.036965,0.890578,0.539757,-0.237558,-0.256942,0.036028,0.807976
3588,0.188854,-1.146600,-0.644782,1.062391,0.995485,0.041089,0.212967,1.040189,-1.146334,0.218228,...,-2.062110,-0.443374,0.118230,-0.205855,0.472956,0.859512,-0.843042,1.118569,0.006060,0.770851
3589,-0.195376,-0.222648,0.269063,-0.578467,1.802839,0.404609,-0.484939,0.027754,0.756843,-0.877412,...,0.278394,-0.533028,0.128055,0.231859,1.099389,-0.186509,0.065184,0.161692,0.165892,0.803347
3590,-0.364305,0.479422,-0.130744,-0.898871,0.610233,-0.514104,0.552136,-0.820763,0.466686,0.510398,...,0.080605,-1.406324,0.049456,0.139205,-0.153477,-0.557338,-1.099208,-0.430376,1.584402,-0.665905


### train model

In [15]:
# Define the XGBoost classifier
xgb_clf = xgb.XGBClassifier(eval_metric='logloss', random_state=88)

# Define the hyperparameter space to search
param_dist = {
    'n_estimators': [25, 50],
    'max_depth': [2, 3],  # lower max_depth to simplify the model
    'learning_rate': [0.01, 0.1],
    'subsample': [0.6, 0.8],
    'colsample_bytree': [0.6, 0.8],
    'gamma': [0, 0.1],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# Create a scorer based on AUC score
auc_scorer = make_scorer(roc_auc_score)

# Set up the random search with cross-validation
random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_dist,
    scoring=auc_scorer,
    n_iter=10,  # Number of iterations for random search
    cv=3,       # Number of folds in cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1   # Use all available cores
)

# Perform the random search
random_search.fit(X_train_processed, y_train)

# Output the best parameters and best score
print("Best parameters found: ", random_search.best_params_)
print("Best AUC score: ", random_search.best_score_)

# Evaluate the model on the train set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_train_processed)[:, 1]
train_auc_score = roc_auc_score(y_train, y_pred_proba)
print("Train AUC score: ", train_auc_score)

# Evaluate the model on the test set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_test_processed)[:, 1]
test_auc_score = roc_auc_score(y_test, y_pred_proba)
print("Test AUC score: ", test_auc_score)

# Evaluate the model on the oot set
best_model = random_search.best_estimator_
y_pred_proba = best_model.predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("TRAIN GINI score: ", round(2*train_auc_score-1,3))
print("Test GINI score: ", round(2*test_auc_score-1,3))
print("OOT GINI score: ", round(2*oot_auc_score-1,3))

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters found:  {'subsample': 0.6, 'reg_lambda': 1.5, 'reg_alpha': 1, 'n_estimators': 50, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.6}
Best AUC score:  0.7144304932948812
Train AUC score:  0.8764051050955646
Test AUC score:  0.8319111273881462
OOT AUC score:  0.8395101955013664
TRAIN GINI score:  0.753
Test GINI score:  0.664
OOT GINI score:  0.679


### compute metrics to monitor drift

In [21]:
def make_bins_from_quantiles(s, q=(0, .05, .1, .2, .3, .4, .5, .6, .7, .8, .9, .95, 1)):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return np.array([0.0, 1.0])
    cuts = np.unique(s.quantile(q, interpolation="linear").values)
    if cuts[0] > s.min(): cuts = np.insert(cuts, 0, s.min())
    if cuts[-1] < s.max(): cuts = np.append(cuts, s.max())
    return np.unique(cuts)

def hist_proportions(values, bin_edges):
    counts, _ = np.histogram(values, bins=bin_edges)
    total = counts.sum()
    return (counts / total).tolist() if total > 0 else [0.0] * (len(bin_edges) - 1)

def series_stats(s):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return {"n": 0, "missing_rate": 1.0, "mean": None, "std": None,
                "min": None, "max": None, "p50": None}
    return {
        "n": int(s.notna().sum()),
        "missing_rate": float(1.0 - s.notna().mean()),
        "mean": float(s.mean()),
        "std": float(s.std(ddof=1)),
        "min": float(s.min()),
        "max": float(s.max()),
        "p50": float(s.quantile(0.5))
    }

# ------------------------------------------------------------------
# 1) Feature names
# ------------------------------------------------------------------
if hasattr(X_train, "columns"):
    feature_names = list(X_train.columns)
else:
    feature_names = [f"f{i}" for i in range(X_train_processed.shape[1])]

# ------------------------------------------------------------------
# 2) Top-3 XGBoost feature importances (gain)
# ------------------------------------------------------------------
booster = best_model.get_booster()
gain_map = booster.get_score(importance_type="gain")

# map booster keys ('f0','f1',...) back to actual names, rank by gain
gain_pairs = [
    (feature_names[j], float(gain_map.get(f"f{j}", 0.0)))
    for j in range(len(feature_names))
]
gain_pairs.sort(key=lambda x: x[1], reverse=True)

# take only the top 3 features
top_features = gain_pairs[:3]


In [23]:
feature_names

['avg_fe_1',
 'avg_fe_2',
 'avg_fe_3',
 'avg_fe_4',
 'avg_fe_5',
 'avg_fe_6',
 'avg_fe_7',
 'avg_fe_8',
 'avg_fe_9',
 'avg_fe_10',
 'avg_fe_11',
 'avg_fe_12',
 'avg_fe_13',
 'avg_fe_14',
 'avg_fe_15',
 'avg_fe_16',
 'avg_fe_17',
 'avg_fe_18',
 'avg_fe_19',
 'avg_fe_20',
 'Num_Fin_Pdts',
 'Debt_to_Salary',
 'Loans_per_Credit_Item',
 'Changed_Credit_Limit',
 'Credit_History_Age_Month',
 'Outstanding_Debt_log']

In [22]:
top_features

[('Outstanding_Debt_log', 59.51771545410156),
 ('Num_Fin_Pdts', 20.24140739440918),
 ('Debt_to_Salary', 16.35113525390625)]

In [24]:
# ------------------------------------------------------------------
# 3) Drift baselines (score + top-3 features)
# ------------------------------------------------------------------
# ---- score baseline
train_scores = best_model.predict_proba(X_train_processed)[:, 1]
score_bins = make_bins_from_quantiles(pd.Series(train_scores))
score_ref_hist = hist_proportions(train_scores, score_bins)

In [31]:
# ---- feature baselines
feature_bins, feature_ref_hists, feature_stats = {}, {}, {}
X_train_df = X_train if hasattr(X_train, "columns") else pd.DataFrame(X_train_processed, columns=feature_names)
for feat, _ in top_features:
    s = pd.to_numeric(X_train_df[feat], errors="coerce")
    bins = make_bins_from_quantiles(s)
    feature_bins[feat] = bins.tolist()
    feature_ref_hists[feat] = hist_proportions(s.values, bins)
    feature_stats[feat] = series_stats(s)

### prepare model artefact to save

In [32]:
model_artefact = {}

model_artefact["features"] = {
    "feature_names": feature_names,
    "n_features": len(feature_names),
}

model_artefact["important_features"] = {
    "names": [name for name, _ in top_features],
    "importance_gain": {name: imp for name, imp in top_features}
}

model_artefact["drift_baseline"] = {
    "score_bins": score_bins.tolist(),
    "score_ref_hist": score_ref_hist,
    "score_baseline_stats": series_stats(pd.Series(train_scores)),
    "top_features": top_features,
    "feature_bins": feature_bins,
    "feature_ref_hists": feature_ref_hists,
    "feature_baseline_stats": feature_stats,
}

model_artefact["drift_baseline"]["score_deciles"] = list(
    map(float, np.quantile(train_scores, np.linspace(0, 1, 11)))
)

model_artefact['model'] = best_model
model_artefact['model_version'] = "credit_model_"+config["model_train_date_str"].replace('-','_')
model_artefact['preprocessing_transformers'] = {}
model_artefact['preprocessing_transformers']['stdscaler'] = transformer_stdscaler
model_artefact['data_dates'] = config
model_artefact['data_stats'] = {}
model_artefact['data_stats']['X_train'] = X_train.shape[0]
model_artefact['data_stats']['X_test'] = X_test.shape[0]
model_artefact['data_stats']['X_oot'] = X_oot.shape[0]
model_artefact['data_stats']['y_train'] = round(y_train.mean(),2)
model_artefact['data_stats']['y_test'] = round(y_test.mean(),2)
model_artefact['data_stats']['y_oot'] = round(y_oot.mean(),2)
model_artefact['results'] = {}
model_artefact['results']['auc_train'] = train_auc_score
model_artefact['results']['auc_test'] = test_auc_score
model_artefact['results']['auc_oot'] = oot_auc_score
model_artefact['results']['gini_train'] = round(2*train_auc_score-1,3)
model_artefact['results']['gini_test'] = round(2*test_auc_score-1,3)
model_artefact['results']['gini_oot'] = round(2*oot_auc_score-1,3)
model_artefact['hp_params'] = random_search.best_params_


pprint.pprint(model_artefact)

{'data_dates': {'model_train_date': datetime.datetime(2024, 6, 1, 0, 0),
                'model_train_date_str': '2024-06-01',
                'oot_end_date': datetime.datetime(2024, 5, 31, 0, 0),
                'oot_period_months': 2,
                'oot_start_date': datetime.datetime(2024, 4, 1, 0, 0),
                'train_test_end_date': datetime.datetime(2024, 3, 31, 0, 0),
                'train_test_period_months': 9,
                'train_test_ratio': 0.8,
                'train_test_start_date': datetime.datetime(2023, 7, 1, 0, 0)},
 'data_stats': {'X_oot': 978,
                'X_test': 899,
                'X_train': 3592,
                'y_oot': np.float64(0.27),
                'y_test': np.float64(0.28),
                'y_train': np.float64(0.28)},
 'drift_baseline': {'feature_baseline_stats': {'Debt_to_Salary': {'max': 11.122651100158691,
                                                                  'mean': 0.7244516611099243,
                                  

### save artefact to model bank

In [33]:
# create model_bank dir
model_bank_directory = "model_bank/"

if not os.path.exists(model_bank_directory):
    os.makedirs(model_bank_directory)

In [34]:
# Full path to the file
file_path = os.path.join(model_bank_directory, model_artefact['model_version'] + '.pkl')

# Write the model to a pickle file
with open(file_path, 'wb') as file:
    pickle.dump(model_artefact, file)

print(f"Model saved to {file_path}")


Model saved to model_bank/credit_model_2024_06_01.pkl


### test load pickle and make model inference

In [35]:
# Load the model from the pickle file
with open(file_path, 'rb') as file:
    loaded_model_artefact = pickle.load(file)

y_pred_proba = loaded_model_artefact['model'].predict_proba(X_oot_processed)[:, 1]
oot_auc_score = roc_auc_score(y_oot, y_pred_proba)
print("OOT AUC score: ", oot_auc_score)

print("Model loaded successfully!")

OOT AUC score:  0.8395101955013664
Model loaded successfully!
